# 【House Prices】特徴量強化とアンサンブルの入口

特徴量を強化し、複数モデルの予測を平均するブレンディングでスコアを底上げします。

> **Kaggle Tutorial** 対応ノートブック（`/learn/intro-competitions/07-house-prices-ensemble`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/house-prices-advanced-regression-techniques を使用し、
# Google Colab やローカル環境では互換スキーマの住宅価格サンプルデータを自動配置します。
import os
import numpy as np
import pandas as pd

def _ensure_house_prices_data():
    target_dir = "/kaggle/input/house-prices-advanced-regression-techniques"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/house-prices-advanced-regression-techniques")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)

    def _make_split(n, start_id, with_target):
        overall_qual = rng.integers(3, 10, size=n)
        first_flr = rng.normal(1160, 320, size=n).clip(450, 3200).round().astype(int)
        second_flr = np.where(rng.random(n) > 0.45, rng.normal(650, 260, size=n).clip(0, 1800), 0).round().astype(int)
        bsmt_sf = rng.normal(1050, 350, size=n).clip(0, 3000).round()
        gr_liv_area = first_flr + second_flr
        garage_cars = rng.choice([0, 1, 2, 3], size=n, p=[0.06, 0.27, 0.55, 0.12])
        year_built = rng.integers(1950, 2010, size=n)
        year_remod = np.maximum(year_built, rng.integers(1960, 2011, size=n))
        yr_sold = rng.integers(2006, 2011, size=n)
        lot_frontage = rng.normal(70, 20, size=n).clip(21, 180).round(1)
        lot_frontage[rng.random(n) < 0.17] = np.nan
        bsmt_With_nan = bsmt_sf.copy()
        if not with_target:
            bsmt_With_nan[0] = np.nan

        neighborhood = rng.choice(["NAmes", "CollgCr", "OldTown", "Edwards", "Somerst", "NridgHt"], size=n)
        ms_zoning = rng.choice(["RL", "RM", "FV", "C (all)"], size=n, p=[0.78, 0.15, 0.05, 0.02]).astype(object)
        if not with_target:
            ms_zoning[0] = "RareZone"  # test のみに現れる未知カテゴリ（handle_unknown="ignore" 検証用）
        kitchen_qual = np.where(overall_qual >= 8, "Ex", np.where(overall_qual >= 6, "Gd", "TA")).astype(object)
        garage_type = np.where(garage_cars == 0, None, rng.choice(["Attchd", "Detchd", "BuiltIn"], size=n).astype(object))

        df = pd.DataFrame({
            "Id": np.arange(start_id, start_id + n),
            "MSSubClass": rng.choice([20, 60, 50, 120], size=n),
            "MSZoning": pd.Series(ms_zoning, dtype=object),
            "LotFrontage": lot_frontage,
            "LotArea": rng.lognormal(9.1, 0.4, size=n).round().astype(int),
            "Neighborhood": neighborhood,
            "OverallQual": overall_qual,
            "OverallCond": rng.integers(4, 9, size=n),
            "YearBuilt": year_built,
            "YearRemodAdd": year_remod,
            "TotalBsmtSF": bsmt_With_nan,
            "1stFlrSF": first_flr,
            "2ndFlrSF": second_flr,
            "GrLivArea": gr_liv_area,
            "FullBath": rng.choice([1, 2, 3], size=n, p=[0.45, 0.50, 0.05]),
            "KitchenQual": pd.Series(kitchen_qual, dtype=object),
            "GarageType": pd.Series(garage_type, dtype=object),
            "GarageCars": garage_cars,
            "YrSold": yr_sold,
        })
        if with_target:
            log_price = (
                10.8
                + 0.14 * overall_qual
                + 0.00028 * gr_liv_area
                + 0.00018 * np.nan_to_num(bsmt_With_nan, nan=0.0)
                + 0.08 * garage_cars
                + 0.003 * (year_built - 1970)
                + np.where(neighborhood == "NridgHt", 0.18, 0.0)
                + rng.normal(0.0, 0.11, size=n)
            )
            df["SalePrice"] = np.expm1(log_price).round().astype(int)
        return df

    _make_split(1460, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(1459, 1461, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"House Prices dataset ready at: {data_dir}")

_ensure_house_prices_data()

## 特徴量を強化する

ドメイン知識から、効きそうな特徴を作ります。House Prices の定番は面積系の合算です。

In [ ]:
def add_features(df):
    df = df.copy()
    # 総面積（地下 + 1 階 + 2 階）
    df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"] + df["2ndFlrSF"]
    # 築年・改装からの経過
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    return df

歪んだ数値特徴は対数変換すると線形モデルに効くことがあります。

## アンサンブルの入口：ブレンディング

異なる性質のモデルの予測を平均すると、単体より安定して精度が上がることがよくあります。
最も簡単なアンサンブル学習が、予測値の加重平均＝ブレンディングです。

In [ ]:
# 強化した特徴量と前処理 Pipeline を使って Ridge と GradientBoostingRegressor を学習します
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = add_features(pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv"))
test = add_features(pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv"))

y = np.log1p(train["SalePrice"])
X = train.drop(columns=["Id", "SalePrice"])
X_test = test.drop(columns=["Id"])

num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("object").columns

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

ridge = Pipeline([("pre", pre), ("ridge", Ridge(alpha=10.0))]).fit(X, y)
gbr = Pipeline([("pre", pre), ("gbr", GradientBoostingRegressor(random_state=42))]).fit(X, y)

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

# それぞれ学習済みとする（前処理 Pipeline 込み）
pred_ridge = np.expm1(ridge.predict(X_test))   # 線形モデル
pred_gbr = np.expm1(gbr.predict(X_test))        # 勾配ブースティング

# 単純平均でブレンド
pred = 0.5 * pred_ridge + 0.5 * pred_gbr

In [ ]:
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": pred})
print("Blended predictions preview:")
submission.head()

線形モデル（ロジスティック回帰の回帰版 Ridge）と勾配ブースティングは得意・不得意が異なるため、
組み合わせると弱点を補い合えます。

> **Tip:** ブレンドの重みは交差検証の OOF 予測で決めると過学習しにくくなります。 まずは 0.5 : 0.5 から試し、CV を見ながら調整しましょう。

> **Note:** 本格的なスタッキング（メタモデルで統合）や勾配ブースティング（GBDT）の本命 （LightGBM / XGBoost）は、次トラック L5・L6 で詳しく扱います。

## 次のステップ

最後に、提出フローの完全理解と、よくある submission 形式エラーのデバッグを学びます。